In [1]:
import torch
import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

# Tắt cảnh báo symlinks của Hugging Face
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Kiểm tra thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device.upper()}")

# Cấu hình Model chung
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

C:\Users\ASUS\miniconda3\envs\vqa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang sử dụng thiết bị: CUDA


In [2]:
print("Đang tải dữ liệu...")
raw_dataset = load_dataset("tomaarsen/setfit-absa-semeval-restaurants")

def group_aspects_for_extractor(hf_dataset):
    """Gom nhóm các aspects thuộc cùng một câu thành danh sách"""
    df = hf_dataset.to_pandas()
    # Gom các khía cạnh (span) theo từng câu (text)
    grouped = df.groupby('text')['span'].apply(list).reset_index()
    return Dataset.from_pandas(grouped)

# Tạo dataset cho Aspect Extractor
ds_ext_train = group_aspects_for_extractor(raw_dataset['train'])
ds_ext_test = group_aspects_for_extractor(raw_dataset['test'])

print(f"Số lượng câu Train (Extractor): {len(ds_ext_train)}")
print(f"Số lượng câu Test (Extractor): {len(ds_ext_test)}")

Đang tải dữ liệu...
Số lượng câu Train (Extractor): 2019
Số lượng câu Test (Extractor): 606


In [3]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['text'], 
        truncation=True, 
        padding='max_length', 
        max_length=128, 
        return_offsets_mapping=True
    )
    
    labels = []
    for i, (text, spans) in enumerate(zip(examples['text'], examples['span'])):
        offsets = tokenized['offset_mapping'][i]
        # Khởi tạo: 0 ('O') cho token chữ, -100 cho special tokens
        label_ids = [0 if (o[0] != o[1]) else -100 for o in offsets]
        
        for span in spans:
            start_idx = 0
            while True:
                start_char = text.lower().find(span.lower(), start_idx)
                if start_char == -1: break
                end_char = start_char + len(span)
                
                first_token = True
                for idx, (s, e) in enumerate(offsets):
                    if s == e: continue # Bỏ qua special tokens
                    if s >= start_char and e <= end_char:
                        if first_token:
                            label_ids[idx] = 1 # B-ASP
                            first_token = False
                        elif label_ids[idx] == 0:
                            label_ids[idx] = 2 # I-ASP
                            
                start_idx = end_char # Tìm tiếp nếu từ đó xuất hiện nhiều lần
                
        labels.append(label_ids)
        
    tokenized['labels'] = labels
    return tokenized

encoded_ext_train = ds_ext_train.map(tokenize_and_align_labels, batched=True, remove_columns=ds_ext_train.column_names)
encoded_ext_test = ds_ext_test.map(tokenize_and_align_labels, batched=True, remove_columns=ds_ext_test.column_names)
print("Dán nhãn thành công!")

Map: 100%|██████████████████████████████████████████████████████████████████| 606/606 [00:00<00:00, 3194.05 examples/s]

Dán nhãn thành công!


In [4]:
# Tải metrics đánh giá
seqeval = evaluate.load("seqeval")
label_list = ["O", "B-ASP", "I-ASP"]

def compute_metrics_ext(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

model_ext = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=3, use_safetensors=True).to(device)

args_ext = TrainingArguments(
    output_dir="./saved_models/extractor_checkpoints",
    num_train_epochs=5,
    learning_rate=2e-5,          # LR an toàn
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=False,                  # TẮT để chống nan
    warmup_ratio=0.1,            # Khởi động mềm
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    dataloader_num_workers=0,
    logging_steps=20,            # Hiện loss đều đặn
    report_to="none"
)

trainer_ext = Trainer(
    model=model_ext,
    args=args_ext,
    train_dataset=encoded_ext_train,
    eval_dataset=encoded_ext_test,
    compute_metrics=compute_metrics_ext,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

trainer_ext.train()
model_ext.save_pretrained("./saved_models/aspect_extractor")
tokenizer.save_pretrained("./saved_models/aspect_extractor")
print("Đã lưu mô hình")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 3531.49it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading 

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.126336,0.107240,0.793103,0.866025,0.827961,0.961809
2,0.085150,0.098839,0.830453,0.883538,0.856173,0.969015
3,0.035817,0.113528,0.858221,0.895797,0.876607,0.971717
4,0.017353,0.127554,0.849338,0.898424,0.873191,0.970906
5,0.015924,0.138740,0.859060,0.896673,0.877464,0.970906


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.43it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.l

Đã lưu mô hình


In [6]:
# 1. Tải tập Train gốc (Có nhãn đầy đủ)
ds_cls_train_full = raw_dataset['train']
ds_cls_test_blind = raw_dataset['test'] # Tập mù, để dành dự đoán sau

# 2. CẮT 15% TẬP TRAIN ĐỂ LÀM TẬP VALIDATION ĐÁNH GIÁ MÔ HÌNH
split_dataset = ds_cls_train_full.train_test_split(test_size=0.15, seed=42)
ds_cls_train = split_dataset['train']
ds_cls_val = split_dataset['test']

print(f"Số lượng câu Train: {len(ds_cls_train)}")
print(f"Số lượng câu Validation: {len(ds_cls_val)}")

def preprocess_classifier(examples):
    tokenized = tokenizer(
        examples['text'], examples['span'], 
        padding="max_length", truncation=True, max_length=128
    )
    
    labels = []
    label_feature = ds_cls_train_full.features['label']
    has_names = hasattr(label_feature, 'names')
    
    for raw_label in examples['label']:
        str_lbl = ""
        # Đưa nhãn về dạng chữ (positive/negative/neutral)
        if isinstance(raw_label, int) or str(raw_label).isdigit():
            lbl_int = int(raw_label)
            if has_names:
                str_lbl = label_feature.names[lbl_int].lower()
            else:
                mapping = {0: 'negative', 1: 'neutral', 2: 'positive', 3: 'neutral'}
                str_lbl = mapping.get(lbl_int, 'neutral')
        else:
            str_lbl = str(raw_label).strip().lower()
            
        # Gán nhãn 0, 1, 2 cho mô hình
        if 'neg' in str_lbl:
            labels.append(0)
        elif 'pos' in str_lbl:
            labels.append(2)
        else:
            labels.append(1)
            
    tokenized['labels'] = labels
    return tokenized

# 3. Mã hóa dữ liệu trên tập Train và Val mới
encoded_cls_train = ds_cls_train.map(preprocess_classifier, batched=True, remove_columns=ds_cls_train.column_names)
encoded_cls_val = ds_cls_val.map(preprocess_classifier, batched=True, remove_columns=ds_cls_val.column_names)

# 4. KHỞI TẠO TRAINER VÀ HUẤN LUYỆN
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics_cls(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

model_cls = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, use_safetensors=True).to(device)

args_cls = TrainingArguments(
    output_dir="./saved_models/classifier_checkpoints",
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=False,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=0,
    logging_steps=20,
    report_to="none"
)

trainer_cls = Trainer(
    model=model_cls,
    args=args_cls,
    train_dataset=encoded_cls_train,
    eval_dataset=encoded_cls_val,     # Sử dụng tập Val đã cắt có đáp án
    compute_metrics=compute_metrics_cls
)

print("\nBẮT ĐẦU TRAIN SENTIMENT CLASSIFIER")
trainer_cls.train()

# Lưu mô hình hoàn chỉnh
model_cls.save_pretrained("./saved_models/sentiment_classifier")
tokenizer.save_pretrained("./saved_models/sentiment_classifier")
print("Đã lưu mô hình Classifier")

Số lượng câu Train: 3139
Số lượng câu Validation: 554


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3923.08it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initializ


BẮT ĐẦU TRAIN SENTIMENT CLASSIFIER


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.672310,0.631636,0.741877,0.650711
2,0.529242,0.593416,0.779783,0.692437
3,0.346892,0.606096,0.777978,0.720402
4,0.186736,0.653371,0.788809,0.736722
5,0.141304,0.671708,0.790614,0.734038


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.58it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.l

Đã lưu mô hình Classifier
